# 🧹 Semana 2: Calidad, Limpieza, Transformación e Ingeniería de Datos

## Objetivos
- Comprender las dimensiones clave de la calidad de datos
- Aplicar técnicas de limpieza: valores faltantes, duplicados y outliers
- Transformar y preprocesar datos: normalización, encoding y discretización
- Crear nuevas características (Feature Engineering)
- Dominar funciones avanzadas de Pandas

## Dataset: Water Quality (Calidad del Agua)
Continuamos con el dataset de calidad del agua de la Semana 1.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
np.random.seed(42)

print("✅ Librerías importadas")

In [ ]:
# Carga del dataset
url = "https://raw.githubusercontent.com/jaquimbayoc7/material-fundamentos_datos/main/data/water_potability.csv"
df = pd.read_csv(url)

# Copia original para comparaciones posteriores
df_original = df.copy()

print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

---
# 1. Fundamentos de la Calidad de Datos

## 1.1 Introducción a la Calidad de Datos

La **calidad de datos** se refiere al grado en que los datos son aptos para su uso previsto en operaciones, análisis y toma de decisiones. Datos de mala calidad pueden llevar a:

- Conclusiones erróneas en análisis estadísticos
- Modelos de Machine Learning con bajo rendimiento
- Decisiones empresariales equivocadas
- Pérdida de credibilidad en reportes

> **"Garbage In, Garbage Out"** — Si los datos de entrada son basura, los resultados también lo serán.

## 1.2 Dimensiones Clave de la Calidad

| Dimensión | Descripción | Ejemplo en nuestro dataset |
|-----------|-------------|---------------------------|
| **Exactitud** | Los datos representan correctamente la realidad | ¿El pH de 14.5 es un valor real? (pH máximo es 14) |
| **Completitud** | Ausencia de valores nulos o faltantes | ¿Cuántos NaN hay en la columna `Sulfate`? |
| **Consistencia** | Coherencia entre diferentes campos | ¿Son coherentes los rangos de cada variable? |
| **Validez** | Conformidad con reglas o formatos definidos | pH debe estar entre 0 y 14 |
| **Unicidad** | No existen registros duplicados | ¿Hay filas repetidas en el dataset? |

## 1.3 Diagnóstico de Problemas

Antes de limpiar, debemos **diagnosticar** los problemas de calidad. Veamos las técnicas más comunes:

In [ ]:
# ═══════════════════════════════════════════════════════════
# DIAGNÓSTICO COMPLETO DE CALIDAD DE DATOS
# ═══════════════════════════════════════════════════════════

print("📊 DIAGNÓSTICO DE CALIDAD DEL DATASET")
print("=" * 55)

# 1. COMPLETITUD — Valores faltantes
print("\n🔍 1. COMPLETITUD — Valores Faltantes")
print("-" * 55)
nulos = df.isnull().sum()
porcentaje = (df.isnull().sum() / len(df) * 100).round(2)
diagnostico = pd.DataFrame({'Nulos': nulos, '% Nulos': porcentaje})
diagnostico = diagnostico[diagnostico['Nulos'] > 0].sort_values('% Nulos', ascending=False)
if len(diagnostico) > 0:
    print(diagnostico)
else:
    print("✅ No hay valores faltantes")

# 2. UNICIDAD — Duplicados
print(f"\n🔍 2. UNICIDAD — Duplicados")
print("-" * 55)
duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {duplicados} ({duplicados/len(df)*100:.2f}%)")

# 3. VALIDEZ — Rangos esperados
print(f"\n🔍 3. VALIDEZ — Verificación de Rangos")
print("-" * 55)
# pH debe estar entre 0 y 14
ph_invalidos = df[(df['ph'] < 0) | (df['ph'] > 14)].shape[0]
print(f"pH fuera de rango [0-14]: {ph_invalidos} registros")
# Potabilidad debe ser 0 o 1
pot_invalidos = df[~df['Potability'].isin([0, 1])].shape[0]
print(f"Potabilidad fuera de {{0, 1}}: {pot_invalidos} registros")

# 4. EXACTITUD — Estadísticas descriptivas para detectar anomalías
print(f"\n🔍 4. EXACTITUD — Resumen Estadístico")
print("-" * 55)
display(df.describe().round(2))

In [ ]:
# Visualización de valores faltantes
plt.figure(figsize=(12, 5))

# Mapa de calor de datos faltantes
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd')
plt.title('Mapa de Valores Faltantes en el Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Variables')
plt.ylabel('Registros')
plt.tight_layout()
plt.show()

---
# 2. Técnicas de Limpieza de Datos (Data Cleaning)

## 2.1 Manejo de Valores Faltantes (Missing Values)

Los valores faltantes (`NaN`, `None`) son uno de los problemas más comunes. Tenemos dos estrategias principales:

1. **Eliminación**: Remover filas o columnas con datos faltantes
2. **Imputación**: Reemplazar los valores faltantes con estimaciones (media, mediana, moda, constante)

In [ ]:
# Identificación de valores nulos
print("🔍 IDENTIFICACIÓN DE VALORES NULOS")
print("=" * 50)

# Métodos de detección
print("\n1. Verificar si hay algún nulo en todo el DataFrame:")
print(f"   df.isnull().values.any() → {df.isnull().values.any()}")

print("\n2. Total de nulos en todo el DataFrame:")
print(f"   df.isnull().sum().sum() → {df.isnull().sum().sum()}")

print("\n3. Nulos por columna:")
print(df.isnull().sum())

print(f"\n4. Filas con al menos un nulo: {df.isnull().any(axis=1).sum()}")
print(f"   Filas completas (sin nulos): {df.dropna().shape[0]}")

In [ ]:
# ── ESTRATEGIA 1: Eliminación ──
print("❌ ESTRATEGIA 1: ELIMINACIÓN")
print("=" * 50)

# Eliminación por filas (se pierden registros)
df_sin_nulos_filas = df.dropna()
print(f"\nEliminación por filas:")
print(f"  Original: {df.shape[0]} filas")
print(f"  Después:  {df_sin_nulos_filas.shape[0]} filas")
print(f"  Perdidas: {df.shape[0] - df_sin_nulos_filas.shape[0]} filas ({(1 - df_sin_nulos_filas.shape[0]/df.shape[0])*100:.1f}%)")

# Eliminación por columnas (se pierden variables)
# Solo eliminar columnas con más del 30% de nulos
umbral = 0.30
cols_a_eliminar = [col for col in df.columns if df[col].isnull().sum()/len(df) > umbral]
print(f"\nColumnas con más de {umbral*100}% nulos: {cols_a_eliminar if cols_a_eliminar else 'Ninguna'}")

In [ ]:
# ── ESTRATEGIA 2: Imputación ──
print("✅ ESTRATEGIA 2: IMPUTACIÓN")
print("=" * 50)

df_imputado = df.copy()

# Columnas con valores nulos
cols_con_nulos = df.columns[df.isnull().any()].tolist()
print(f"Columnas a imputar: {cols_con_nulos}")

for col in cols_con_nulos:
    media = df[col].mean()
    mediana = df[col].median()
    
    # Verificar la asimetría para elegir la mejor estrategia
    asimetria = df[col].skew()
    
    if abs(asimetria) > 1:  # Distribución muy asimétrica → usar mediana
        df_imputado[col].fillna(mediana, inplace=True)
        metodo = 'MEDIANA'
    else:  # Distribución relativamente simétrica → usar media
        df_imputado[col].fillna(media, inplace=True)
        metodo = 'MEDIA'
    
    print(f"  {col:20s} → Asimetría: {asimetria:+.2f} → Imputado con {metodo} ({media:.2f} / {mediana:.2f})")

print(f"\n✅ Nulos restantes: {df_imputado.isnull().sum().sum()}")

In [ ]:
# Comparación visual: antes y después de la imputación
fig, axes = plt.subplots(1, len(cols_con_nulos), figsize=(5*len(cols_con_nulos), 4))
if len(cols_con_nulos) == 1:
    axes = [axes]

for ax, col in zip(axes, cols_con_nulos):
    ax.hist(df[col].dropna(), bins=30, alpha=0.5, color='steelblue', label='Original', density=True)
    ax.hist(df_imputado[col], bins=30, alpha=0.5, color='coral', label='Imputado', density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend()

plt.suptitle('Comparación de Distribuciones: Original vs Imputado', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.2 Tratamiento de Datos Duplicados

Los registros duplicados pueden sesgar análisis estadísticos y el entrenamiento de modelos. Es importante detectarlos y decidir cómo manejarlos.

In [ ]:
# Detección de duplicados
print("🔍 DETECCIÓN DE DUPLICADOS")
print("=" * 50)

# Duplicados exactos (todas las columnas iguales)
duplicados_exactos = df_imputado.duplicated().sum()
print(f"Duplicados exactos: {duplicados_exactos}")

# Duplicados basados en subconjunto de columnas
duplicados_parciales = df_imputado.duplicated(subset=['ph', 'Hardness', 'Solids']).sum()
print(f"Duplicados por [ph, Hardness, Solids]: {duplicados_parciales}")

# Mostrar filas duplicadas si existen
if duplicados_exactos > 0:
    print("\nEjemplo de filas duplicadas:")
    display(df_imputado[df_imputado.duplicated(keep=False)].head(10))

In [ ]:
# Eliminación de duplicados
filas_antes = df_imputado.shape[0]
df_imputado = df_imputado.drop_duplicates()
filas_despues = df_imputado.shape[0]

print(f"Filas antes: {filas_antes}")
print(f"Filas después: {filas_despues}")
print(f"Eliminadas: {filas_antes - filas_despues}")

## 2.3 Detección y Manejo de Outliers (Valores Atípicos)

Los **outliers** son valores que se desvían significativamente del resto de los datos. Pueden ser:
- **Legítimos**: valores reales pero extremos
- **Errores**: errores de medición o captura

### Métodos de detección:
1. **IQR (Rango Intercuartílico)**: Valores fuera de [Q1 - 1.5×IQR, Q3 + 1.5×IQR]
2. **Z-score (Puntuación Z)**: Valores con |Z| > 3 (más de 3 desviaciones estándar)

In [ ]:
# ── MÉTODO 1: Detección por IQR ──
print("📐 DETECCIÓN DE OUTLIERS — MÉTODO IQR")
print("=" * 55)

def detectar_outliers_iqr(dataframe, columna):
    """Detecta outliers usando el método IQR."""
    Q1 = dataframe[columna].quantile(0.25)
    Q3 = dataframe[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    outliers = dataframe[(dataframe[columna] < limite_inferior) | (dataframe[columna] > limite_superior)]
    return outliers, limite_inferior, limite_superior

cols_numericas = df_imputado.select_dtypes(include=[np.number]).columns.tolist()
cols_numericas = [c for c in cols_numericas if c != 'Potability']

print(f"{'Columna':20s} {'Outliers':>10s} {'% del total':>12s} {'Lím. Inf':>12s} {'Lím. Sup':>12s}")
print("-" * 70)

for col in cols_numericas:
    outliers, li, ls = detectar_outliers_iqr(df_imputado, col)
    pct = len(outliers) / len(df_imputado) * 100
    print(f"{col:20s} {len(outliers):10d} {pct:11.2f}% {li:12.2f} {ls:12.2f}")

In [ ]:
# ── MÉTODO 2: Detección por Z-score ──
print("📐 DETECCIÓN DE OUTLIERS — MÉTODO Z-SCORE")
print("=" * 55)

from scipy.stats import zscore

# Calcular Z-scores para todas las columnas numéricas
z_scores = np.abs(zscore(df_imputado[cols_numericas], nan_policy='omit'))

# Outliers: valores con Z-score > 3
umbral_z = 3
print(f"Umbral Z-score: |Z| > {umbral_z}\n")

print(f"{'Columna':20s} {'Outliers (Z>3)':>15s} {'% del total':>12s}")
print("-" * 50)

for i, col in enumerate(cols_numericas):
    n_outliers = (z_scores[:, i] > umbral_z).sum()
    pct = n_outliers / len(df_imputado) * 100
    print(f"{col:20s} {n_outliers:15d} {pct:11.2f}%")

In [ ]:
# Visualización de outliers con boxplots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Detección Visual de Outliers (Boxplots)', fontsize=16, fontweight='bold')

for i, col in enumerate(cols_numericas):
    ax = axes[i // 3, i % 3]
    sns.boxplot(data=df_imputado, y=col, ax=ax, color='steelblue')
    ax.set_title(col, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Manejo de outliers: Imputación con límites IQR (winsorización)
print("🔧 MANEJO DE OUTLIERS — WINSORIZACIÓN")
print("=" * 50)

df_limpio = df_imputado.copy()

for col in cols_numericas:
    Q1 = df_limpio[col].quantile(0.25)
    Q3 = df_limpio[col].quantile(0.75)
    IQR = Q3 - Q1
    li = Q1 - 1.5 * IQR
    ls = Q3 + 1.5 * IQR
    
    # Reemplazar outliers por los límites
    antes = ((df_limpio[col] < li) | (df_limpio[col] > ls)).sum()
    df_limpio[col] = df_limpio[col].clip(lower=li, upper=ls)
    despues = ((df_limpio[col] < li) | (df_limpio[col] > ls)).sum()
    
    if antes > 0:
        print(f"  {col}: {antes} outliers winsorizados")

print(f"\n✅ Dataset limpio: {df_limpio.shape}")

---
# 3. Transformación y Preprocesamiento de Datos

## 3.1 Normalización y Estandarización de Datos Numéricos

Muchos algoritmos de ML requieren que las variables estén en la misma escala:

- **Normalización (Min-Max)**: Escala los datos al rango [0, 1]
  $$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

- **Estandarización (Z-score)**: Transforma para tener media = 0 y desviación estándar = 1
  $$X_{std} = \frac{X - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ── Normalización Min-Max ──
print("📊 NORMALIZACIÓN MIN-MAX [0, 1]")
print("=" * 50)

scaler_minmax = MinMaxScaler()
df_normalizado = pd.DataFrame(
    scaler_minmax.fit_transform(df_limpio[cols_numericas]),
    columns=cols_numericas
)
print("Estadísticas después de normalización:")
display(df_normalizado.describe().round(3))

In [ ]:
# ── Estandarización Z-score ──
print("📊 ESTANDARIZACIÓN Z-SCORE (media=0, std=1)")
print("=" * 50)

scaler_std = StandardScaler()
df_estandarizado = pd.DataFrame(
    scaler_std.fit_transform(df_limpio[cols_numericas]),
    columns=cols_numericas
)
print("Estadísticas después de estandarización:")
display(df_estandarizado.describe().round(3))

In [ ]:
# Comparación visual: Original vs Normalizado vs Estandarizado
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

col_ejemplo = 'Hardness'

axes[0].hist(df_limpio[col_ejemplo], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title(f'{col_ejemplo} — Original', fontweight='bold')

axes[1].hist(df_normalizado[col_ejemplo], bins=30, color='coral', edgecolor='white')
axes[1].set_title(f'{col_ejemplo} — Normalizado [0,1]', fontweight='bold')

axes[2].hist(df_estandarizado[col_ejemplo], bins=30, color='mediumseagreen', edgecolor='white')
axes[2].set_title(f'{col_ejemplo} — Estandarizado (Z)', fontweight='bold')

plt.suptitle('Comparación de Escalado', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.2 Codificación de Variables Categóricas

Los modelos de ML trabajan con números, por lo que las variables categóricas deben ser codificadas:

- **Label Encoding**: Asigna un número entero a cada categoría. Ideal para variables **ordinales** (con orden).
- **One-Hot Encoding**: Crea una columna binaria por cada categoría. Ideal para variables **nominales** (sin orden).

In [ ]:
# Crear una variable categórica de ejemplo basada en pH
# para demostrar los métodos de codificación

df_encoding = df_limpio.copy()

# Variable ordinal: Nivel de pH (bajo, medio, alto)
df_encoding['pH_nivel'] = pd.cut(
    df_encoding['ph'],
    bins=[0, 6.5, 7.5, 14],
    labels=['Ácido', 'Neutro', 'Alcalino']
)

# Variable nominal: Clasificación de calidad
df_encoding['calidad_agua'] = pd.cut(
    df_encoding['Turbidity'],
    bins=[0, 3, 4, float('inf')],
    labels=['Buena', 'Aceptable', 'Mala']
)

print("Variables categóricas creadas:")
print(f"\npH_nivel:\n{df_encoding['pH_nivel'].value_counts()}")
print(f"\ncalidad_agua:\n{df_encoding['calidad_agua'].value_counts()}")

In [ ]:
# ── Label Encoding (para variable ordinal) ──
from sklearn.preprocessing import LabelEncoder

print("🏷️ LABEL ENCODING (Variable Ordinal: pH_nivel)")
print("=" * 50)

le = LabelEncoder()
df_encoding['pH_nivel_encoded'] = le.fit_transform(df_encoding['pH_nivel'])

# Mapeo de etiquetas
mapeo = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Mapeo: {mapeo}")
print(df_encoding[['pH_nivel', 'pH_nivel_encoded']].drop_duplicates())

In [ ]:
# ── One-Hot Encoding (para variable nominal) ──
print("🔥 ONE-HOT ENCODING (Variable Nominal: calidad_agua)")
print("=" * 55)

# Método con pandas (get_dummies)
df_onehot = pd.get_dummies(df_encoding[['calidad_agua']], prefix='calidad', dtype=int)
print("\nResultado del One-Hot Encoding:")
display(df_onehot.head(10))

print(f"\nColumnas creadas: {df_onehot.columns.tolist()}")

## 3.3 Discretización (Binning)

La **discretización** convierte variables numéricas continuas en categorías discretas ("bins" o intervalos). Útil cuando:
- Se necesita simplificar el análisis
- Se quieren crear grupos significativos
- Algunos algoritmos funcionan mejor con datos categóricos

In [ ]:
# Discretización de la variable Hardness (Dureza del agua)
print("📦 DISCRETIZACIÓN — Dureza del Agua")
print("=" * 50)

# Método 1: Bins de igual ancho (cut)
df_bins = df_limpio.copy()
df_bins['Hardness_bin_equal'] = pd.cut(df_bins['Hardness'], bins=4,
                                        labels=['Blanda', 'Moderada', 'Dura', 'Muy Dura'])

# Método 2: Bins de igual frecuencia (qcut)
df_bins['Hardness_bin_quantile'] = pd.qcut(df_bins['Hardness'], q=4,
                                            labels=['Q1', 'Q2', 'Q3', 'Q4'])

print("\nBins de igual ancho:")
print(df_bins['Hardness_bin_equal'].value_counts().sort_index())

print("\nBins de igual frecuencia (cuartiles):")
print(df_bins['Hardness_bin_quantile'].value_counts().sort_index())

In [ ]:
# Visualización de la discretización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_bins['Hardness_bin_equal'].value_counts().sort_index().plot(kind='bar', ax=axes[0],
    color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
axes[0].set_title('Bins de Igual Ancho', fontweight='bold')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=0)

df_bins['Hardness_bin_quantile'].value_counts().sort_index().plot(kind='bar', ax=axes[1],
    color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
axes[1].set_title('Bins de Igual Frecuencia (Cuartiles)', fontweight='bold')
axes[1].set_ylabel('Frecuencia')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Discretización de la Dureza del Agua', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 4. Ingeniería de Características (Feature Engineering)

## 4.1 Creación de Nuevas Variables

La ingeniería de características consiste en crear **nuevas variables informativas** a partir de las existentes para mejorar el rendimiento de los modelos.

In [ ]:
# Creación de nuevas características relevantes para calidad del agua
print("🔧 INGENIERÍA DE CARACTERÍSTICAS")
print("=" * 50)

df_features = df_limpio.copy()

# 1. Ratio entre sólidos y conductividad
df_features['ratio_solidos_conductividad'] = df_features['Solids'] / df_features['Conductivity']

# 2. Indicador de pH neutro (entre 6.5 y 8.5 según OMS)
df_features['ph_neutral'] = ((df_features['ph'] >= 6.5) & (df_features['ph'] <= 8.5)).astype(int)

# 3. Índice de contaminación orgánica (carbono + trihalometanos)
df_features['indice_contaminacion'] = (
    df_features['Organic_carbon'] * 0.5 + df_features['Trihalomethanes'] * 0.5
)

# 4. Logaritmo de Solids (para reducir asimetría)
df_features['log_solids'] = np.log1p(df_features['Solids'])

# 5. Interacción entre Hardness y Sulfate
df_features['hardness_x_sulfate'] = df_features['Hardness'] * df_features['Sulfate']

print("Nuevas variables creadas:")
nuevas_cols = ['ratio_solidos_conductividad', 'ph_neutral', 'indice_contaminacion',
               'log_solids', 'hardness_x_sulfate']
display(df_features[nuevas_cols].describe().round(2))

## 4.2 Extracción de Características

Consiste en derivar información útil de datos complejos. Aunque nuestro dataset no tiene fechas ni texto, demostraremos ambas técnicas con ejemplos simulados.

In [ ]:
# Simulación de extracción de características de fechas
print("📅 EXTRACCIÓN DE CARACTERÍSTICAS — FECHAS")
print("=" * 50)

# Simular fechas de muestreo
np.random.seed(42)
fechas = pd.date_range(start='2023-01-01', periods=len(df_features), freq='2h')
df_features['fecha_muestreo'] = fechas[:len(df_features)]

# Extraer componentes de la fecha
df_features['anio'] = df_features['fecha_muestreo'].dt.year
df_features['mes'] = df_features['fecha_muestreo'].dt.month
df_features['dia'] = df_features['fecha_muestreo'].dt.day
df_features['dia_semana'] = df_features['fecha_muestreo'].dt.dayofweek  # 0=Lunes
df_features['hora'] = df_features['fecha_muestreo'].dt.hour
df_features['trimestre'] = df_features['fecha_muestreo'].dt.quarter

print("Características extraídas de la fecha:")
display(df_features[['fecha_muestreo', 'anio', 'mes', 'dia', 'dia_semana', 'hora', 'trimestre']].head())

---
# 5. Aplicación Práctica con Librerías de Python

## 5.1 Manipulación Avanzada con Pandas

### `apply()`, `map()` y `transform()`

In [ ]:
# ── apply() — Aplicar una función a filas o columnas ──
print("🔧 FUNCIÓN apply()")
print("=" * 50)

df_pandas = df_limpio.copy()

# apply() sobre una columna (Series)
def clasificar_ph(ph):
    """Clasifica el pH del agua."""
    if ph < 6.5:
        return 'Ácido'
    elif ph <= 8.5:
        return 'Neutro'
    else:
        return 'Alcalino'

df_pandas['ph_clase'] = df_pandas['ph'].apply(clasificar_ph)
print("Clasificación del pH con apply():")
print(df_pandas['ph_clase'].value_counts())

# apply() sobre filas (axis=1) — crear un score de riesgo
def score_riesgo(fila):
    """Calcula un score de riesgo hídrico basado en múltiples variables."""
    score = 0
    if fila['ph'] < 6.5 or fila['ph'] > 8.5:
        score += 1
    if fila['Turbidity'] > 4:
        score += 1
    if fila['Chloramines'] > 8:
        score += 1
    return score

df_pandas['riesgo'] = df_pandas.apply(score_riesgo, axis=1)
print(f"\nDistribución del score de riesgo:")
print(df_pandas['riesgo'].value_counts().sort_index())

In [ ]:
# ── map() — Mapear valores a otros valores ──
print("🗺️ FUNCIÓN map()")
print("=" * 50)

# map() con diccionario
df_pandas['potabilidad_texto'] = df_pandas['Potability'].map({
    0: 'No Potable',
    1: 'Potable'
})
print("Mapeo de Potabilidad:")
print(df_pandas[['Potability', 'potabilidad_texto']].drop_duplicates())

In [ ]:
# ── transform() — Transformaciones que mantienen el índice ──
print("🔄 FUNCIÓN transform()")
print("=" * 50)

# transform() es ideal para operaciones con groupby que devuelven
# un valor por cada fila original

# Media del pH por grupo de potabilidad (broadcast al tamaño original)
df_pandas['ph_media_grupo'] = df_pandas.groupby('Potability')['ph'].transform('mean')

# Diferencia respecto a la media del grupo
df_pandas['ph_diff_grupo'] = df_pandas['ph'] - df_pandas['ph_media_grupo']

print("pH original vs media del grupo vs diferencia:")
display(df_pandas[['ph', 'Potability', 'ph_media_grupo', 'ph_diff_grupo']].head(10))

### `groupby()` — Agregación y Resumen de Datos

In [ ]:
# Agregación con groupby()
print("📊 AGREGACIÓN CON groupby()")
print("=" * 50)

# Estadísticas por grupo de potabilidad
resumen = df_pandas.groupby('Potability')[cols_numericas].agg(['mean', 'std', 'min', 'max'])
print("Resumen estadístico por Potabilidad:")
display(resumen.round(2))

In [ ]:
# Múltiples funciones de agregación personalizadas
resumen_custom = df_pandas.groupby('Potability').agg(
    ph_promedio=('ph', 'mean'),
    ph_mediana=('ph', 'median'),
    dureza_max=('Hardness', 'max'),
    turbidez_promedio=('Turbidity', 'mean'),
    n_muestras=('ph', 'count')
).round(2)

resumen_custom.index = ['No Potable', 'Potable']
print("Resumen personalizado:")
display(resumen_custom)

### Combinación de DataFrames: `merge`, `join`, `concat`

In [ ]:
# ── concat() — Concatenar DataFrames ──
print("🔗 CONCATENACIÓN CON concat()")
print("=" * 50)

# Dividir en dos partes y recombinar (simulación)
df_parte1 = df_limpio.iloc[:100]  # Primeras 100 filas
df_parte2 = df_limpio.iloc[100:200]  # Filas 100-200

# Concatenar verticalmente (apilar filas)
df_concat = pd.concat([df_parte1, df_parte2], axis=0, ignore_index=True)
print(f"Parte 1: {df_parte1.shape} + Parte 2: {df_parte2.shape} = Concatenado: {df_concat.shape}")

# Concatenar horizontalmente (agregar columnas)
df_info1 = df_limpio[['ph', 'Hardness']].head(5)
df_info2 = df_limpio[['Turbidity', 'Potability']].head(5)
df_concat_h = pd.concat([df_info1, df_info2], axis=1)
print(f"\nConcatenación horizontal:")
display(df_concat_h)

In [ ]:
# ── merge() — Unión tipo SQL ──
print("🔗 MERGE (Unión tipo SQL)")
print("=" * 50)

# Crear tablas de ejemplo para demostrar merge
# Tabla de muestras
muestras = pd.DataFrame({
    'muestra_id': [1, 2, 3, 4, 5],
    'ph': [7.2, 6.8, 8.1, 5.5, 7.0],
    'estacion_id': ['E01', 'E02', 'E01', 'E03', 'E02']
})

# Tabla de estaciones de monitoreo
estaciones = pd.DataFrame({
    'estacion_id': ['E01', 'E02', 'E03'],
    'ubicacion': ['Río Bogotá', 'Laguna Fúquene', 'Embalse Tominé'],
    'departamento': ['Cundinamarca', 'Cundinamarca', 'Cundinamarca']
})

# Inner merge
df_merged = pd.merge(muestras, estaciones, on='estacion_id', how='inner')
print("Resultado del merge (inner join):")
display(df_merged)

# Tipos de merge:
print("\nTipos de merge disponibles:")
print("  - inner: solo filas con coincidencia en ambas tablas")
print("  - left:  todas las filas de la tabla izquierda")
print("  - right: todas las filas de la tabla derecha")
print("  - outer: todas las filas de ambas tablas")

---
## Conclusiones

En esta semana aprendimos el pipeline completo de preparación de datos:

1. **Calidad de Datos**: Diagnosticamos problemas usando las 5 dimensiones (exactitud, completitud, consistencia, validez, unicidad)
2. **Limpieza**: Tratamos valores faltantes (imputación), duplicados y outliers (IQR, Z-score)
3. **Transformación**: Aplicamos normalización Min-Max, estandarización Z-score, Label/One-Hot Encoding y discretización
4. **Ingeniería de Características**: Creamos nuevas variables informativas y extrajimos componentes de fechas
5. **Pandas Avanzado**: Dominamos `apply()`, `map()`, `transform()`, `groupby()` y combinación de DataFrames

**Próxima semana:** Análisis de correlación, supuestos estadísticos (homoscedasticidad, multicolinealidad) y formulación de hipótesis.